# Lab 04: Transformer Block

**Module 04** | Read `notes/04-transformers.pdf` before running this notebook.

Build one transformer encoder block from scratch: multi-head self-attention, feed-forward network, residual connections, and layer normalization.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


Transformers replace recurrence with self-attention: every position can directly attend to every other position in one parallel pass. A full encoder stacks many identical layers; here we implement **one** encoder layer from scratch to see the internals, multi-head attention, feed-forward network, residual connections, and layer normalization.

Understanding this block is the key to reading larger models (BERT, GPT, ViT) because they reuse the same pattern at depth.


Scaled dot-product attention computes compatibility scores between queries and keys, scales by √d_k for stable gradients, softmaxes into weights, and mixes values. Multi-head attention runs several attention operations in parallel on projected subspaces, then concatenates and projects back to model dimension.

Each head learns different relational patterns (local vs. broad dependencies) without increasing sequence length.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

BATCH_SIZE = 4
SEQ_LEN = 16
D_MODEL = 64
NUM_HEADS = 4
D_FF = 256
DROPOUT = 0.1


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, seq_len, d = x.shape
        qkv = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [
            tensor.view(b, seq_len, self.num_heads, self.d_head).transpose(1, 2)
            for tensor in qkv
        ]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = self.dropout(F.softmax(scores, dim=-1))
        mixed = attn @ v
        mixed = mixed.transpose(1, 2).contiguous().view(b, seq_len, d)
        return self.out(mixed)


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerEncoderLayerScratch(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_out = self.self_attn(x)
        x = self.norm1(x + self.dropout(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x


layer = TransformerEncoderLayerScratch(D_MODEL, NUM_HEADS, D_FF, DROPOUT).to(device)
print(layer)


The forward pass below feeds a random tensor shaped `[batch, sequence, features]`. We print tensor shapes at each stage so you can verify that sequence length and batch dimension are preserved, a single encoder layer maps `(B, T, D)` to `(B, T, D)`.

Stacking many such layers (not done here) builds hierarchical representations without changing rank.


In [ ]:
x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
print(f"Input shape:           {tuple(x.shape)}")

with torch.no_grad():
    attn_only = layer.self_attn(x)
    print(f"After self-attention:  {tuple(attn_only.shape)}")

    after_norm1 = layer.norm1(x + layer.dropout(attn_only))
    print(f"After residual+norm1:  {tuple(after_norm1.shape)}")

    ff_out = layer.ff(after_norm1)
    print(f"After feed-forward:    {tuple(ff_out.shape)}")

    y = layer(x)
    print(f"Layer output shape:    {tuple(y.shape)}")

assert y.shape == x.shape, "Encoder layer must preserve (batch, seq, d_model)"
print("\nShape check passed, ready to stack into a full encoder.")
